In [1]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import liana as li
from liana.method import cellphonedb, cellchat
from tqdm import tqdm
#liana dotplot returns a ggsave object...
from plotnine import ggsave, ggplot
import warnings
warnings.filterwarnings("ignore")

In [2]:
H = ["IndividualMiceData/GF_HF_6B_CellPhoneDB.h5ad","IndividualMiceData/GF_HF_13L_CellPhoneDB.h5ad"]
D = ["IndividualMiceData/GF_HFVHC_5A_CellPhoneDB.h5ad","IndividualMiceData/GF_HFVHC_16L_CellPhoneDB.h5ad"]
data = {}
data["H1"] = sc.read_h5ad(H[0])
data["H2"] = sc.read_h5ad(H[1])
data["D1"] = sc.read_h5ad(D[0])
data["D2"] = sc.read_h5ad(D[1])

In [4]:
score = {}
for t in ["H","D"]:
    for i in [1,2]:
        data_key = t+str(i)
        data_curr = data[data_key]
        for s in ['cDC1s','cDC2s','Mig. cDCs','pDCs']:
            for l in ['H2-Aa','H2-Ab1']:
                for r in ['Cd4', 'Lag3']:
                    x = data_curr.uns["cpdb_res"][data_curr.uns["cpdb_res"].ligand == l]\
                    [data_curr.uns["cpdb_res"].receptor == r]\
                    [data_curr.uns["cpdb_res"].target == 'T cells']\
                    [data_curr.uns["cpdb_res"].source == s]
                    if x.empty:
                        score[(data_key, s, l, r)] = 0
                    else:
                        score[(data_key, s, l, r)] = x.lr_probs.iloc[0]

In [5]:
import numpy as np
# Extract unique combinations of source, ligand, and receptor
unique_combinations = set([(item[1], item[2], item[3]) for item in score.keys()])

for source, ligand, receptor in unique_combinations:
    healthy_scores = []
    diseased_scores = []
    healthy_labels = []
    diseased_labels = []

    for key, value in score.items():
        mouse, s, l, r = key
        if s == source and l == ligand and r == receptor:
            if mouse.startswith('H'):
                healthy_scores.append(value)
                healthy_labels.append(mouse)
            elif mouse.startswith('D'):
                diseased_scores.append(value)
                diseased_labels.append(mouse)

    # Create the plot
    plt.figure(figsize=(6, 4))
    x_healthy = np.zeros_like(healthy_scores)
    x_diseased = np.ones_like(diseased_scores)

    plt.scatter(x_healthy, healthy_scores, color='green', label='Healthy')
    plt.scatter(x_diseased, diseased_scores, color='red', label='Diseased')

    # Annotate individual points
    for i, txt in enumerate(healthy_labels):
        plt.annotate(txt, (x_healthy[i], healthy_scores[i]), textcoords="offset points", xytext=(0,5), ha='center')
    for i, txt in enumerate(diseased_labels):
        plt.annotate(txt, (x_diseased[i], diseased_scores[i]), textcoords="offset points", xytext=(0,5), ha='center')

    plt.xticks([0, 1], ['Healthy', 'Diseased'])
    plt.ylabel('Interaction Score')
    plt.title(f'{source} -> T cells\nLigand: {ligand}, Receptor: {receptor}')
    plt.legend()
    plt.tight_layout()

    # Save the plot with a descriptive name
    filename = f"figures/Interactions/GF_CellPhoneDB_interaction_{source.replace(' ', '_')}_{ligand}_{receptor}.png"
    plt.savefig(filename)
    plt.close()

print("Plots have been saved as PNG files.")

Plots have been saved as PNG files.
